# Granite based on local behavior (effects) for Bike Rental Predictions

This notebook leverages GRANITE to yield regional explanations on the BikeSharing dataset using local effects. We first look at minimizing disagreement due to interactions (pure vs. full) and then to minimize disagreement due to distributional influence (marginal vs. conditional)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Local imports
from granite.data import get_data_bike
from granite.decompositions import get_components_brute_force

### Input Features

The BikeSharing dataset aims to predict the hourly count of bike rentals between years 2011
and 2012 in Washington state, based on time and weather features. It involves 17K instances
and 10 features.

First load the data and a `Features` object that stores information about the various features. 
Note that the data **X** must **always** be a numerical numpy array. Categorical features are assumed 
to have been ordinally encoded.

In [ ]:
# Built-in function for the BikeSharing dataset
X, y, features = get_data_bike()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Make months and weekdays smaller for the plots (optional)
features.feature_objs[1].cats = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
features.feature_objs[4].cats = ["Sun", "Mon", "Tue", "Wed", "Thu", "Fri", "Sat"]
features.summary()
cat_features = [0, 1, 3, 4, 5, 6]

In [ ]:
from utils import train_gbt_bikesharing

# Fit the model, or load it if cached in models/
model = train_gbt_bikesharing(X_train, y_train)
test_preds = model.predict(X_test)
print(f" RMSE of the model : {root_mean_squared_error(test_preds, y_test):.2f}")
print(f" Std dev of the target : {y_test.std():.2f}")

## Disagreement due to interactions: Marginal Pure-Full Effects (individual feature influence)

We can understand how an individual features impact the model response by computing their pure and full marginal effects.
We first compute the Anchored decomposition.

In [ ]:
from granite.decompositions import get_components_tree
# Compute the H^k matrices using a subset of 1000 training points
background = X_test[:1000]
decomposition = get_components_tree(model, background, background, features=features, anchored=True)

In [ ]:
# The keys are tuples encoding subsets of features
display(decomposition.keys())
# The () key is the intercept
display(decomposition[()].shape)
# The key (i,) is the main effect of feature i in the Anchored Decomposition
display(decomposition[(0,)].shape)

In [ ]:
from granite.plots import partial_dependence_plot, set_ticks_categorical_features

# Plot the PDP and the ICE curves
partial_dependence_plot(decomposition, background, features, centered=True, figsize=(20, 10), n_cols=5)
plt.savefig("../figures/bike_pure_full_global_all.pdf", bbox_inches="tight")

plt.show()

**Hour and temperature have the highers heterogeneity (besides workingday)**. We can study them more closely.

In [ ]:
# Plot the PDP and the ICE curves
fig, ax = partial_dependence_plot(
    decomposition,
    background,
    features,
    idxs=[2, 7],
    centered=True,
    figsize=(5, 2.5)
)

# Postprocessing
# If ax is an array, iterate over it
if isinstance(ax, np.ndarray):
    for i, a in enumerate(ax.ravel()):
        a.set_ylim(-400, 500)
        if i == 0:
            a.set_ylabel("effect")
else:
    ax.set_ylim(-400, 500)
    ax.set_ylabel("effect")



plt.savefig("../figures/bike_pure_full_global_main.pdf", bbox_inches='tight', dpi=300)
plt.show()
# Save the entire figure (all selected plots) as a PDF
##import matplotlib.pyplot as plt
#plt.savefig("../figures/bike_pure_full_global_main.pdf", bbox_inches="tight")  # tight layout
#plt.close()  # closes the figure to free memory

In these plots, dark lines are PDPs (pure-marginal-local) explantions while the thin lines are the ICEs (full-marginal-local). Difference between the two are
indicators of feature interactions. We minimize pure-vs-full disagreements cause by 
interactions within a subset $U\subseteq 2^{D}$ of the feature powerset set $2^{D}$.

- Fixing $U=\{\{0\}, \{1\}, \ldots \{d\}\}$ minimizes all interactions
- Fixing $U=\{\{7\}\}$ minimizes all interactions involving feature $7$.

In [ ]:
from granite.fd_trees import FDTree
from granite.experiments import get_marginal_pure_vs_full_loss_fn

# Option 1: We minimize all interactions
U = [(i,) for i in range(len(features))]
features_to_split_upon = [i for i in range(len(features))]

# # Option 2: We minimize all interactions involving `temp`
# U = [(7,)]
# features_to_split_upon = [i for i in range(len(features)) if not i == 7]

# We penalize all pure-vs-full effects of each feature, which ammounts to minimizing all interactions within the model
loss_fn = get_marginal_pure_vs_full_loss_fn(
    decomposition=decomposition,
    U=U
)
tree = FDTree(
    max_depth=3,
    features=features.select(features_to_split_upon),
    alpha=0.1,
    save_losses=True
)
tree.fit(background[:, features_to_split_upon], loss_fn=loss_fn)
tree.print(verbose=True)

In [ ]:
# Using regional backgrounds
larger_background = X_test
regions = tree.predict(larger_background[:, features_to_split_upon])
print(np.unique(regions, return_counts=True))
rules = tree.rules()
print(rules)

In [ ]:
# We compute regional decompositions
regional_backgrounds = [[]] * tree.n_regions
regional_decomposition = [[]] * tree.n_regions
regional_shap = [[]] * tree.n_regions
for r in range(tree.n_regions):
    regional_backgrounds[r] = larger_background[regions==r]
    # Regional Decomposition
    regional_decomposition[r] = get_components_tree(
                                        model,
                                        regional_backgrounds[r],
                                        regional_backgrounds[r],
                                        features,
                                        anchored=True
                                    )

**We plot the regional effects for all features -- disagreement is highly reduced, however, there seems to be still some left, particularly in temperature and in hour**

In [ ]:
from granite.plots import plot_legend

# Regional h_i and SHAP values
partial_dependence_plot(regional_decomposition, regional_backgrounds, features, centered=True, figsize=(20, 10), n_cols=5)
plt.savefig("../figures/bike_pure_full_regional_all.pdf", bbox_inches='tight')

plot_legend(rules, ncol=4)
plt.savefig("../figures/bike_pure_full_regional_all_legend.pdf", bbox_inches='tight')

plt.show()

In [ ]:
from granite.plots import plot_legend

# Regional h_i and SHAP values
fig, ax = partial_dependence_plot(
    regional_decomposition,
    regional_backgrounds,
    features, 
    idxs=[2, 7],
    centered=True,
    figsize=(5, 2.5),
    n_cols = 2
)


# TODO: adjust ax and fig if necessary here
# If ax is an array, iterate over it
if isinstance(ax, np.ndarray):
    for i, a in enumerate(ax.ravel()):
        a.set_ylim(-400, 500)
        if i == 0:
            a.set_ylabel("effect")
else:
    ax.set_ylim(-400, 500)
    ax.set_ylabel("effect")
plt.savefig("../figures/bike_pure_full_regional_main.pdf", bbox_inches='tight')

plot_legend(rules, ncol=4)
plt.savefig("../figures/bike_pure_full_regional_main_legend.pdf", bbox_inches='tight')

plt.show()

**We split on depth deeper and see that disagreement is further reduced, at the cost
of interpretability**

In [ ]:
# Option 1: We minimize all interactions
U = [(i,) for i in range(len(features))]
features_to_split_upon = [i for i in range(len(features))]

# # Option 2: We minimize all interactions involving `temp`
# U = [(7,)]
# features_to_split_upon = [i for i in range(len(features)) if not i == 7]

# We penalize all pure-vs-full effects of each feature, which ammounts to minimizing all interactions within the model
loss_fn = get_marginal_pure_vs_full_loss_fn(
    decomposition=decomposition,
    U=U
)
tree = FDTree(
    max_depth=4,
    features=features.select(features_to_split_upon),
    alpha=0.05,
    save_losses=True
)
tree.fit(background[:, features_to_split_upon], loss_fn=loss_fn)
tree.print(verbose=True)

In [ ]:
# Using regional backgrounds
larger_background = X_test
regions = tree.predict(larger_background[:, features_to_split_upon])
print(np.unique(regions, return_counts=True))
rules = tree.rules()
print(rules)

In [ ]:
# We compute regional decompositions
regional_backgrounds = [[]] * tree.n_regions
regional_decomposition = [[]] * tree.n_regions
regional_shap = [[]] * tree.n_regions
for r in range(tree.n_regions):
    regional_backgrounds[r] = larger_background[regions==r]
    # Regional Decomposition
    regional_decomposition[r] = get_components_tree(
                                        model,
                                        regional_backgrounds[r],
                                        regional_backgrounds[r],
                                        features,
                                        anchored=True
                                    )

In [ ]:
# Regional h_i and SHAP values
partial_dependence_plot(
    regional_decomposition,
    regional_backgrounds,
    features,
    centered=True,
    figsize=(15, 10),
)
plt.savefig("../figures/bike_pure_full_regional_all_deep.pdf", bbox_inches='tight')
plt.show()

plot_legend(rules, ncol=4)
plt.savefig("../figures/bike_pure_full_regional_all_deep_legend.pdf", bbox_inches='tight')
plt.show()

## Pure vs. Full for Interaction influence measures

We were able to reduce disagreements between hour and temperature a bit more by splitting deeper, however, such interactions between numeric features are more difficult to handle with partitioning since we need many splits. 

To handle this problem, we could accept to introduce order-2 interactions into our explanations. Now, the pure-vs-full disagreements would only be cause by strong interactions of order-3 or more. Since order-2 interactions remain interpretable (they can be plotted on a screen), we could potentially extract insightful regional
explanations.

In [ ]:
# Compute the H^k matrices using a subset of 1000 test points
background = X_test[:1000]

# Compute the Interventional Decomposition
decomposition = get_components_brute_force(
    lambda X: model.predict(X),  # use predict instead of calling the model directly
    background,
    background,
    features,
    interactions=2,
    show_bar=True
)
display(decomposition.keys())

In [ ]:
from granite.fd_trees import FDTree
from granite.experiments import get_marginal_pure_vs_full_loss_fn

# We minimize the pure-full interactions involving features x0 and x1
loss_fn = get_marginal_pure_vs_full_loss_fn(
    decomposition=decomposition,
    U=[(i,) for i in range(len(features))],
)
features_to_split_upon = list(range(10))
tree = FDTree(
    features.select(features_to_split_upon),
    max_depth=1,
    save_losses=True,
    branching_per_node=2,
    alpha=0.1
)
tree.fit(background[:, features_to_split_upon], loss_fn=loss_fn)
tree.print(verbose=True)

In [ ]:
# Using regional backgrounds
larger_background = X_test[:3000]
regions = tree.predict(larger_background[:, features_to_split_upon])
display(regions[:10])
n_regions = tree.n_regions
rules = tree.rules()
display(rules)

In [ ]:
# We compute regional decompositions
regional_backgrounds = [[]] * tree.n_regions
regional_decomposition = [[]] * tree.n_regions
regional_shap = [[]] * tree.n_regions
for r in range(tree.n_regions):
    regional_backgrounds[r] = larger_background[regions==r]
    # Regional Decomposition
    regional_decomposition[r] = get_components_brute_force(
                    lambda X: model.predict(X),  # use predict instead of calling the model directly
                    regional_backgrounds[r],
                    regional_backgrounds[r],
                    features,
                    interactions=2
                )

In [ ]:
from matplotlib.colors import CenteredNorm
from granite.plots import get_red_white_blue_cmap

cmap = get_red_white_blue_cmap()

n_regions = tree.n_regions
# Regional h_i and SHAP values
max_abs_val = max(*[np.abs(regional_decomposition[r][(2, 7)].mean(-1)).max() for r in range(n_regions)])
fig, axs = plt.subplots(1, n_regions, figsize=(5, 2.5))
for r in range(n_regions):
    sc = axs[r].scatter(
        regional_backgrounds[r][:, 2],  # x-axis
        regional_backgrounds[r][:, 7],  # y-axis
        c=regional_decomposition[r][(2, 7)].mean(-1),
        s=10,
        marker="s",
        cmap=cmap,
        norm=CenteredNorm(0, halfrange=max_abs_val)
    )
    axs[r].set_xlabel("hr")

    # ✅ Set ylabel only on the first suÏbplot
    if r == 0:
        axs[r].set_ylabel("temp")
    else:
        axs[r].set_ylabel("")             # remove text
        axs[r].tick_params(labelleft=False)  # hide tick labels too

    axs[r].set_title(rules[r])

# fig.colorbar(cmap, ax=axs[r])


plt.savefig("../figures/bike_pure_full_interaction_regional.pdf", bbox_inches='tight')
plt.show()


This plot presents the regional pair-wise interaction between temperature and hour.
It appears that the effect of higher temperature is reduced early in the morning and late at night. During the day, the effect of high temperature is increased at commute time on workingdays and during the afternoon on non-working days. These trends required a lot of splits to be faithfully represented with univariate feature attributions. Allowing for pair-wise interactions within the explanation allows us to highlight these trends using a single split.

## Disagreement due to distribution: Conditional vs. Marginal Pure Effects (individual feature influence)



In [ ]:
# create bins for all features
from granite.utils import create_bins_for_data

_, binned_features = create_bins_for_data(
    x=background,
    cat_feature_indices=cat_features,
    n_bins_numerical=5,
)

for i, bins in enumerate(binned_features):
    print(f"Feature {i}: Binned Indices Sample = {binned_features[i][:10]}")


In [ ]:
from math import ceil
from granite.utils import decomposition_to_R
from granite.experiments import get_marginal_conditional_effects

def plot_pdp_vs_mplot(
    feature_ids_to_plot,
    decomposition,
    n_cols,
    ylim,
    figsize,
):
    # Specify which features to plot
    n_features = len(feature_ids_to_plot)
    R = decomposition_to_R(decomposition)

    n_rows = ceil(n_features / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()  # flatten to make indexing easier

    for i, feature_id in enumerate(feature_ids_to_plot):
        marginal_effect, conditional_effect = get_marginal_conditional_effects(
            binned_features[feature_id],
            R[(feature_id,)]
        )
        argsort_x_i = np.argsort(background[:, feature_id])

        ax = axes[i]  # pick the subplot for this feature
        ax.plot(background[argsort_x_i, feature_id], marginal_effect[argsort_x_i], 'k-', label="Marginal")
        ax.plot(background[argsort_x_i, feature_id], conditional_effect[argsort_x_i], 'r--', label="Conditional")
        ax.grid("on")

        feature_obj = features.feature_objs[feature_id]

        # Set categorical ticks if applicable
        if feature_obj.type in ["bool", "ordinal", "nominal"]:
            set_ticks_categorical_features(ax, feature_obj, cat_name_threshold=10, degree=45)
        ax.set_xlabel(feature_obj.name)
        if i % n_cols == 0:
            ax.set_ylabel("effect")

        ax.set_ylim(*ylim)
        if i == 0:
            ax.legend(framealpha=1)

    # Turn off any unused axes
    for j in range(n_features, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    return fig, axes


In [ ]:
plot_pdp_vs_mplot(range(10), decomposition, n_cols=5, ylim=(0, 500), figsize=(20, 8))

plt.savefig("../figures/bike_marg_cond_global_all.pdf", bbox_inches='tight')
plt.show()


In [ ]:
plot_pdp_vs_mplot([7, 8], decomposition, n_cols=1, ylim=(0, 500), figsize=(3.5, 5))

plt.savefig("../figures/bike_marg_cond_global_all.pdf", bbox_inches='tight')
plt.show()

In [ ]:
from granite.experiments import get_marginal_vs_conditional_pure_loss_fn


loss_fn = get_marginal_vs_conditional_pure_loss_fn(
    decomposition=decomposition,
    U = [(i,) for i in range(len(features))],
    binned_features=[binned_features[i] for i in range(len(features))]
)
features_to_split_upon = [i for i in range(len(features))]
tree = FDTree(
    features.select(features_to_split_upon), 
    max_depth=2,
    save_losses=True, 
    branching_per_node=3, 
    alpha=0.05
)
tree.fit(background[:, features_to_split_upon], loss_fn=loss_fn)
tree.print(verbose=True)

In [ ]:
larger_background = X_test
_, binned_features = create_bins_for_data(
    x=larger_background,
    cat_feature_indices=cat_features,
    n_bins_numerical=5,
)
regions = tree.predict(larger_background[:, features_to_split_upon])
print(np.unique(regions, return_counts=True))
rules = tree.rules()

In [ ]:
regional_decompositions = [[]] * tree.n_regions
regional_backgrounds = [[]] * tree.n_regions
for r in range(tree.n_regions):
    regional_backgrounds[r] = larger_background[regions == r]
    regional_decompositions[r] = get_components_tree(
        model, 
        regional_backgrounds[r],
        regional_backgrounds[r],
        features,
        anchored=True
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from math import ceil

def plot_regional_mplot_pdp(
    regional_decompositions,
    regional_backgrounds,
    binned_features,
    background,
    features,
    rules,
    n_cols=4,
    cat_name_threshold=10,
    feature_idxs=None,
    figsize=None,  # ✅ New parameter
    ylim: tuple[float, float] | None = None # ✅ New parameter
):
    if feature_idxs is None:
        feature_idxs = list(range(len(features)))
    n_features = len(feature_idxs)
    n_rows = ceil(n_features / n_cols)

    # ✅ Use user-provided figsize or default fallback
    if figsize is None:
        figsize = (5 * n_cols, 4 * n_rows)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    axes = axes.flatten()

    # Color palette for regions
    colors = plt.cm.tab10.colors

    for i, feature_id in enumerate(feature_idxs):
        ax = axes[i]
        feature_obj = features.feature_objs[feature_id]

        # x-range for numeric features
        x_min, x_max = background[:, feature_id].min(), background[:, feature_id].max()

        for r in range(len(regional_decompositions)):
            R = regional_decompositions[r][(feature_id,)] + regional_decompositions[r][()]
            marginal_effect, conditional_effect = get_marginal_conditional_effects(
                binned_features[feature_id][regions == r], R
            )
            X_region = regional_backgrounds[r]
            argsort_x_i = np.argsort(X_region[:, feature_id])
            color = colors[r % len(colors)]

            # Marginal = solid
            ax.plot(
                X_region[argsort_x_i, feature_id],
                marginal_effect[argsort_x_i],
                linestyle='-',
                color=color
            )
            # Conditional = dashed
            ax.plot(
                X_region[argsort_x_i, feature_id],
                conditional_effect[argsort_x_i],
                linestyle='--',
                color=color
            )

        # Handle categorical features
        if feature_obj.type in ["bool", "ordinal", "nominal"]:
            set_ticks_categorical_features(ax, feature_obj, cat_name_threshold=10, degree=45)
        else:
            ax.set_xlim(x_min, x_max)

        if ylim is not None:
            ax.set_ylim(*ylim)
        ax.set_xlabel(feature_obj.name)
        ax.set_ylabel("")
        ax.grid('on')

    # Remove unused subplots
    for i in range(n_features, n_rows * n_cols):
        fig.delaxes(axes[i])

    # Shared legend for marginal vs conditional
    style_handles = [
        plt.Line2D([0], [0], color='black', linestyle='-', lw=2, label='Marginal'),
        plt.Line2D([0], [0], color='black', linestyle='--', lw=2, label='Conditional'),
    ]

    # fig.legend(handles=style_handles, loc='lower right', ncol=2, handlelength=3)

    plt.tight_layout()
    return fig, axes  # ✅ return handles for further customization


def plot_region_legend(rules, colors, fig, max_cols=6, y_offset=-0.05):
    """
    Add a region legend below a figure.

    Parameters
    ----------
    rules : list of str
        Names of the regions.
    colors : list of colors
        Colors corresponding to each region.
    fig : matplotlib.figure.Figure
        The figure to attach the legend to.
    max_cols : int, default=6
        Maximum number of columns in the legend row.
    y_offset : float, default=-0.05
        Vertical position relative to the figure (negative is below the axes).
    """
    n_regions = len(rules)
    ncol = min(n_regions, max_cols)

    handles = [plt.Line2D([0], [0], color=colors[i % len(colors)], lw=2, label=rules[i])
                for i in range(n_regions)]

    fig.legend(
        handles=handles,
        title="Regions",
        title_fontsize=14,
        fontsize=12,
        loc='lower center',
        bbox_to_anchor=(0.5, y_offset),
        bbox_transform=fig.transFigure,
        ncol=ncol,
        handlelength=3
    )







In [ ]:
plot_regional_mplot_pdp(
    regional_decompositions=regional_decompositions,
    regional_backgrounds=regional_backgrounds,
    binned_features=binned_features,
    background=background,
    features=features,
    rules=rules,
    n_cols=5,
    ylim=(0, 500)
)
plt.savefig("../figures/bike_marg_cond_regional_all.pdf", bbox_inches='tight')

colors = plt.cm.tab10.colors
plot_region_legend(rules=rules, colors=colors, fig=plt.gcf(), max_cols=4, y_offset=-0.1)
plt.savefig("../figures/bike_marg_cond_regional_all_legend.pdf", bbox_inches='tight')

plt.show()  # now shows both plots and the legend

In [ ]:
plot_regional_mplot_pdp(
    regional_decompositions=regional_decompositions,
    regional_backgrounds=regional_backgrounds,
    binned_features=binned_features,
    background=background,
    features=features,
    rules=rules,
    n_cols=1,
    feature_idxs=[7, 8],
    figsize=(3.5, 5),
    ylim=(0, 400)
)

plt.savefig("../figures/bike_marg_cond_regional_main.pdf", bbox_inches='tight')
plt.tight_layout(rect=[0, 0.18, 1, 1])  # leave more space for the legend

plot_region_legend(rules=rules, colors=colors, fig=plt.gcf(), max_cols=4, y_offset=-0.05)
plt.savefig("../figures/bike_marg_cond_regional_main_legend.pdf", bbox_inches='tight')
plt.show()